In [1]:
!pip install -U scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 17.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import skfuzzy as fuzz
import skfuzzy.control as ctrl

**2.11** HỆ THỐNG GIÁ TIỀN GRAB-BIKE

In [19]:
distance=ctrl.Antecedent(np.arange(0, 51, 1), 'distance')
traffic=ctrl.Antecedent(np.arange(0, 101, 1), 'traffic')
demand=ctrl.Antecedent(np.arange(0, 101, 1), 'demand')
weather=ctrl.Antecedent(np.arange(0, 11, 1), 'weather')
rating=ctrl.Antecedent(np.arange(0, 6, 1), 'rating')
punctuality=ctrl.Antecedent(np.arange(0, 101, 1), 'punctuality')
price=ctrl.Consequent(np.arange(0, 101, 1), 'price')
bonus=ctrl.Consequent(np.arange(0, 101, 1), 'bonus')

# Khoảng cách (km)
distance['short'] = fuzz.trapmf(distance.universe, [0, 0, 3, 5])
distance['medium'] = fuzz.trimf(distance.universe, [3, 10, 15])
distance['long'] = fuzz.trimf(distance.universe, [6, 15, 25])
distance['very_long'] = fuzz.trapmf(distance.universe, [15, 30, 50, 50])

# Tình trạng giao thông (%)
traffic['low'] = fuzz.trimf(traffic.universe, [0, 0, 30])
traffic['medium'] = fuzz.trimf(traffic.universe, [20, 50, 70])
traffic['high'] = fuzz.trapmf(traffic.universe, [60, 80, 100, 100])

# Mức cầu (%)
demand['low'] = fuzz.trimf(demand.universe, [0, 0, 40])
demand['medium'] = fuzz.trimf(demand.universe, [20, 50, 70])
demand['high'] = fuzz.trapmf(demand.universe, [60, 80, 100, 100])

# Điều kiện khí hậu (Quy ước: 0-3 Tốt, 4-7 Trung bình, 8-10 Xấu)
weather['good'] = fuzz.trimf(weather.universe, [0, 0, 4])
weather['moderate'] = fuzz.trimf(weather.universe, [3, 5, 7])
weather['bad'] = fuzz.trimf(weather.universe, [6, 10, 10])

# Đánh giá khách hàng (1-5 sao)
rating['poor'] = fuzz.trimf(rating.universe, [1, 1, 2.5])
rating['average'] = fuzz.trimf(rating.universe, [2, 3.5, 4.5])
rating['good'] = fuzz.trimf(rating.universe, [3.5, 5, 5])

# Đúng giờ (%)
punctuality['late'] = fuzz.trimf(punctuality.universe, [0, 0, 50])
punctuality['on_time'] = fuzz.trimf(punctuality.universe, [40, 60, 80])
punctuality['soon'] = fuzz.trimf(punctuality.universe, [70, 100, 100])

# Giá
price['low'] = fuzz.trimf(price.universe, [0, 0, 30])
price['medium'] = fuzz.trimf(price.universe, [25, 50, 75])
price['high'] = fuzz.trimf(price.universe, [60, 80, 90])
price['very_high'] = fuzz.trimf(price.universe, [80, 100, 100])

# Thưởng
bonus['none'] = fuzz.trimf(bonus.universe, [0, 0, 10])
bonus['few'] = fuzz.trimf(bonus.universe, [10, 30, 50])
bonus['moderate'] = fuzz.trimf(bonus.universe, [40, 60, 80])
bonus['high'] = fuzz.trimf(bonus.universe, [70, 100, 100])

rules = [
    ctrl.Rule(distance['short'] & traffic['low'] & demand['low'], price['low']), # Luật 1
    ctrl.Rule(distance['short'] & traffic['medium'] & demand['high'], price['medium']), # Luật 2
    ctrl.Rule(distance['medium'] & traffic['high'] & demand['high'], price['high']), # Luật 3
    ctrl.Rule(distance['long'] & traffic['medium'] & weather['good'], price['medium']), # Luật 4
    ctrl.Rule(distance['long'] & traffic['high'] & weather['bad'], price['very_high']), # Luật 5
    ctrl.Rule(distance['very_long'] & traffic['high'] & demand['high'], price['very_high']), #Luật 6
    ctrl.Rule(distance['medium'] & traffic['low'] & demand['low'], price['medium']), # Luật 7
    ctrl.Rule(distance['short'] & traffic['high'] & weather['bad'], price['high']), # Luật 8
    ctrl.Rule(distance['very_long'] & weather['bad'], price['very_high']), # Luật 9
    ctrl.Rule(distance['medium'] & traffic['medium'] & weather['moderate'], price['medium']), # Luật 10
    ctrl.Rule(rating['good'] & punctuality['soon'], bonus['high']), # Luật 11
    ctrl.Rule(rating['average'] & punctuality['on_time'], bonus['moderate']), # Luật 12
    ctrl.Rule(rating['poor'] & punctuality['late'], bonus['none']), # Luật 13
    ctrl.Rule(distance['long'] & traffic['high'] & punctuality['on_time'], bonus['high']), # Luật 14
    ctrl.Rule(distance['medium'] & traffic['high'] & rating['good'], bonus['moderate']), # Luật 15
    ctrl.Rule(rating['good'] & punctuality['late'], bonus['none']), # Luật 16
    ctrl.Rule(distance['very_long'] & weather['bad'] & rating['good'], bonus['high']), # Luật 17
    ctrl.Rule(distance['short'] & rating['average'] & punctuality['on_time'], bonus['few']), # Luật 18
    ctrl.Rule(distance['long'] & traffic['high'] & punctuality['late'], bonus['few']), # Luật 19
    ctrl.Rule(distance['medium'] & weather['moderate'] & rating['good'], bonus['moderate']) # Luật 20
]

grab_ctrl = ctrl.ControlSystem(rules)
grab_sim = ctrl.ControlSystemSimulation(grab_ctrl)

grab_sim.input['distance'] = 20  # 20km
grab_sim.input['traffic'] = 85   # Kẹt xe nặng
grab_sim.input['demand'] = 90    # Nhu cầu cao
grab_sim.input['weather'] = 9    # Trời bão
grab_sim.input['rating'] = 4.8   # Tài xế tốt
grab_sim.input['punctuality'] = 95 # Rất đúng giờ

grab_sim.compute()

print(f"Giá dự kiến: {grab_sim.output['price']:.2f}")
print(f"Điểm thưởng dự kiến: {grab_sim.output['bonus']:.2f}")

Giá dự kiến: 92.22
Điểm thưởng dự kiến: 89.76


**2.12** CHIẾN LƯỢC CHIẾT KHẤU CHO KHÁCH HÀNG Ở CÁC CỬA HÀNG SHOPEE

In [20]:
rating=ctrl.Antecedent(np.arange(0, 6, 1), 'rating')
volume=ctrl.Antecedent(np.arange(0, 101, 1), 'volume')
profit=ctrl.Antecedent(np.arange(0, 101, 1), 'profit')
event=ctrl.Antecedent(np.arange(0, 11, 1), 'event')
competitors=ctrl.Antecedent(np.arange(0, 101, 1), 'competitors')
discount=ctrl.Consequent(np.arange(0, 71, 1), 'discount')

# Đánh giá (1 - 5 sao)
rating['low'] = fuzz.trapmf(rating.universe, [0, 0, 3.5, 4])
rating['medium']=fuzz.trimf(rating.universe, [3.5, 4, 4.5])
rating['high'] = fuzz.trapmf(rating.universe, [4, 4.5, 6, 6])

# Khối lượng bán hàng
volume['low'] = fuzz.trapmf(volume.universe, [0, 0, 20, 35])
volume['medium']=fuzz.trimf(volume.universe, [20, 50, 80])
volume['high'] = fuzz.trapmf(volume.universe, [65, 85, 100, 100])

# Biên lợi nhuận
profit['low'] = fuzz.trapmf(profit.universe, [0, 0, 20, 35])
profit['medium']=fuzz.trimf(profit.universe, [20, 50, 80])
profit['high'] = fuzz.trapmf(profit.universe, [65, 85, 100, 100])

# Sự kiện theo mùa
event['none'] = fuzz.trapmf(event.universe, [0, 0, 2, 4])
event['medium']=fuzz.trimf(event.universe, [3, 5, 7])
event['high'] = fuzz.trapmf(event.universe, [6, 8, 11, 11])

# Giảm giá của đối thủ cạnh tranh
competitors['low'] = fuzz.trapmf(competitors.universe, [0, 0, 20, 35])
competitors['medium']=fuzz.trimf(competitors.universe, [20, 50, 80])
competitors['high'] = fuzz.trapmf(competitors.universe, [65, 85, 100, 100])

# Tỷ lệ phần trăm chiết khấu (%)
discount['very_low'] = fuzz.trapmf(discount.universe, [0, 0, 3, 6])
discount['low']=fuzz.trimf(discount.universe, [5, 7.5, 10])
discount['medium']=fuzz.trimf(discount.universe, [10, 15, 20])
discount['high']=fuzz.trimf(discount.universe, [20, 30, 40])
discount['very_high'] = fuzz.trapmf(discount.universe, [40, 45, 70, 70])

rules = [
    ctrl.Rule(rating['high'] & volume['high'] & profit['high'], discount['very_low']), # Luật 1
    ctrl.Rule(rating['low'] & volume['low'] & profit['high'], discount['high']), # Luật 2
    ctrl.Rule(event['high'] & competitors['high'], discount['very_high']), # Luật 3
    ctrl.Rule(rating['medium'] & volume['medium'] & profit['medium'], discount['medium']), # Luật 4
    ctrl.Rule(competitors['low'] & profit['low'] & volume['high'], discount['very_low']), # Luật 5
    ctrl.Rule(rating['low'] & event['none'], discount['medium']), # Luật 6
    ctrl.Rule(volume['low'] & profit['low'], discount['high']) # Luật 7
]

shopee_ctrl = ctrl.ControlSystem(rules)
shopee_sim = ctrl.ControlSystemSimulation(shopee_ctrl)

shopee_sim.input['rating'] = 4.3 # Trung bình
shopee_sim.input['volume'] = 60 # Trung bình
shopee_sim.input['profit'] = 30 # Thấp
shopee_sim.input['event'] = 8 # Cao
shopee_sim.input['competitors'] = 70 # Cao

shopee_sim.compute()

print(f"Discount dự kiến: {shopee_sim.output['discount']:.2f}")

Discount dự kiến: 44.25


**2.13** KẾ HOẠCH CHIẾN LƯỢC BÁN HÀNG CỦA SHOPEE DÀNH CHO CÁC CỬA HÀNG

In [23]:
demand=ctrl.Antecedent(np.arange(0, 11, 1), 'demand')
comp_pressure=ctrl.Antecedent(np.arange(0, 11, 1), 'comp_pressure')
rating=ctrl.Antecedent(np.arange(0, 5.1, 0.1), 'rating')
price=ctrl.Antecedent(np.arange(0, 11, 1), 'price')
profit=ctrl.Antecedent(np.arange(0, 11, 1), 'profit')
seasonal_demand=ctrl.Antecedent(np.arange(0, 11, 1), 'seasonal_demand')
discount=ctrl.Consequent(np.arange(0, 71, 1), 'discount')

# Nhu cầu sản phẩm
demand['low'] = fuzz.trapmf(demand.universe, [0, 0, 3, 5])
demand['medium']=fuzz.trimf(demand.universe, [4, 5, 7])
demand['high'] = fuzz.trapmf(demand.universe, [6, 8, 10, 10])

# Áp lực định giá đối thủ cạnh tranh
comp_pressure['low'] = fuzz.trapmf(comp_pressure.universe, [0, 0, 3, 5])
comp_pressure['medium']=fuzz.trimf(comp_pressure.universe, [4, 5, 7])
comp_pressure['high'] = fuzz.trapmf(comp_pressure.universe, [6, 8, 10, 10])

# Đánh giá cửa hàng
rating['low'] = fuzz.trapmf(rating.universe, [0, 0, 3.5, 4])
rating['medium'] = fuzz.trimf(rating.universe, [3.5, 4, 4.5])
rating['high'] = fuzz.trapmf(rating.universe, [4, 4.5, 4.9, 5.1])

# Biên lợi nhuận
profit['low'] = fuzz.trapmf(profit.universe, [0, 0, 1, 4])
profit['medium']=fuzz.trimf(profit.universe, [3, 5, 7])
profit['high'] = fuzz.trapmf(profit.universe, [6, 8, 10, 10])

# Nhu cầu theo mùa
seasonal_demand['none'] = fuzz.trapmf(seasonal_demand.universe, [0, 0, 3, 5])
seasonal_demand['medium']=fuzz.trimf(seasonal_demand.universe, [4, 5, 7])
seasonal_demand['high'] = fuzz.trapmf(seasonal_demand.universe, [6, 8, 10, 10])

# Chiết khấu
discount['very_low'] = fuzz.trapmf(discount.universe, [0, 0, 3, 6])
discount['low']=fuzz.trimf(discount.universe, [4, 7, 11])
discount['medium']=fuzz.trimf(discount.universe, [10, 15, 20])
discount['high']=fuzz.trimf(discount.universe, [20, 30, 40])
discount['very_high'] = fuzz.trapmf(discount.universe, [40, 45, 60, 70])

rules = [
    ctrl.Rule(demand['high'] & comp_pressure['low'] & profit['low'], discount['very_low']), # Luật 1
    ctrl.Rule(demand['low'] & comp_pressure['high'] & profit['high'], discount['high']), # Luật 2
    ctrl.Rule(rating['high'] & profit['high'] & seasonal_demand['high'], discount['medium']), # Luật 3 (FIXED: profit['medium'] changed to profit['high'])
    ctrl.Rule(comp_pressure['high'] & seasonal_demand['high'] & profit['high'], discount['very_high']), # Luật 4
    ctrl.Rule(rating['low'] & demand['medium'] & profit['low'], discount['medium']), # Luật 5
    ctrl.Rule(demand['high'] & seasonal_demand['none'] & comp_pressure['low'], discount['very_low']), # Luật 6
    ctrl.Rule(profit['high'] & comp_pressure['medium'] & seasonal_demand['medium'], discount['medium']) # Luật 7
]

shopee_ctrl = ctrl.ControlSystem(rules)
shopee_sim = ctrl.ControlSystemSimulation(shopee_ctrl)

# Sản phẩm: Đồng hồ xa xỉ thủ công
shopee_sim.input['demand'] = 8 # Cao
shopee_sim.input['comp_pressure'] = 6 # Trung bình
shopee_sim.input['rating'] = 4.2 # Trung bình
shopee_sim.input['profit'] = 8 # Cao
shopee_sim.input['seasonal_demand'] = 8 # Cao

shopee_sim.compute()

print(f"Discount dự kiến: {shopee_sim.output['discount']:.2f}")

Discount dự kiến: 15.00


**2.14** TỐI ƯU HÓA THỜI GIAN GIAO HÀNG VÀ TĂNG THU NHẬP CHO TÀI XẾ

In [25]:
density=ctrl.Antecedent(np.arange(0, 11, 1), 'density')
urgency=ctrl.Antecedent(np.arange(0, 11, 1), 'urgency')
curload=ctrl.Antecedent(np.arange(0, 11, 1), 'curload')
traffic=ctrl.Antecedent(np.arange(0, 11, 1), 'traffic')
profit=ctrl.Antecedent(np.arange(0, 11, 1), 'profit')
ordernum=ctrl.Consequent(np.arange(0, 11, 1), 'ordernum')
priority=ctrl.Consequent(np.arange(0, 11, 1), 'priority')

# Mật độ đơn hàng:
density['low'] = fuzz.trapmf(density.universe, [0, 0, 3, 5])
density['medium'] = fuzz.trimf(density.universe, [4, 5, 7])
density['high'] = fuzz.trapmf(density.universe, [6, 8, 10, 10])

# Giao hàng khẩn cấp:
urgency['low'] = fuzz.trapmf(urgency.universe, [0, 0, 3, 5])
urgency['medium'] = fuzz.trimf(urgency.universe, [4, 5, 7])
urgency['high'] = fuzz.trapmf(urgency.universe, [6, 8, 10, 10])

# Tải trọng hiện tại
curload['low'] = fuzz.trapmf(curload.universe, [0, 0, 3, 5])
curload['medium'] = fuzz.trimf(curload.universe, [4, 5, 7])
curload['high'] = fuzz.trapmf(curload.universe, [6, 8, 10, 10])

# Tình trạng giao thông
traffic['low'] = fuzz.trapmf(traffic.universe, [0, 0, 4, 6])
traffic['medium'] = fuzz.trimf(traffic.universe, [3, 5, 8])
traffic['high'] = fuzz.trapmf(traffic.universe, [6, 8, 10, 10])

# Lợi nhuận trên mỗi lần giao
profit['low'] = fuzz.trapmf(profit.universe, [0, 0, 3, 5])
profit['medium'] = fuzz.trimf(profit.universe, [4, 5, 7])
profit['high'] = fuzz.trapmf(profit.universe, [6, 8, 10, 10])

# Số lượng đơn hàng cần kết hợp
ordernum['few'] = fuzz.trapmf(ordernum.universe, [0, 0, 3, 5])
ordernum['some'] = fuzz.trimf(ordernum.universe, [4, 5, 7])
ordernum['many'] = fuzz.trapmf(ordernum.universe, [6, 8, 10, 10])

# Ưu tiên giao hàng
priority['low'] = fuzz.trapmf(priority.universe, [0, 0, 3, 5])
priority['medium'] = fuzz.trimf(priority.universe, [4, 5, 7])
priority['high'] = fuzz.trapmf(priority.universe, [6, 8, 10, 10])

rules = [
    ctrl.Rule(density['high'] & curload['low'] & traffic['low'], ordernum['many']), # Luật 1
    ctrl.Rule(density['medium'] & traffic['high'] & urgency['medium'], ordernum['few']), # Luật 2
    ctrl.Rule(curload['high'] & density['high'] & profit['medium'], ordernum['some']), # Luật 3
    ctrl.Rule(density['low'] & urgency['high'] & traffic['medium'], ordernum['few']), # Luật 4
    ctrl.Rule(profit['high'] & traffic['high'] & urgency['high'], ordernum['few']), # Luật 5
    ctrl.Rule(urgency['high'] & profit['high'], priority['high']), # Luật 6
    ctrl.Rule(urgency['medium'] & traffic['medium'], priority['medium']), # Luật 7
    ctrl.Rule(urgency['low'] & density['high'] & profit['low'], priority['low']) # Luật 8
]

deliver_ctrl = ctrl.ControlSystem(rules)
deliver_sim = ctrl.ControlSystemSimulation(deliver_ctrl)

deliver_sim.input['density'] = 8    # Cao
deliver_sim.input['urgency'] = 6    # Trung bình
deliver_sim.input['curload'] = 3    # Thấp
deliver_sim.input['traffic'] = 5    # Trung bình
deliver_sim.input['profit'] = 5     # Trung bình

deliver_sim.compute()

print(f"Kết hợp đơn hàng: {deliver_sim.output['ordernum']:.2f}")
print(f"Mức độ ưu tiên: {deliver_sim.output['priority']:.2f}")

Kết hợp đơn hàng: 8.24
Mức độ ưu tiên: 5.39
